# fractional-stride-zero-insertion — worked example 1: Zero Insertion for Stride-3 ConvTranspose in 1-D

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `fractional-stride-zero-insertion`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

A ConvTranspose2d with stride `s` is equivalent to inserting `s - 1` zeros between every adjacent pair of input elements, then applying a stride-1 convolution. This zero-dilated tensor has length `(L - 1) * s + 1` for an input of length `L`. The inserted zeros create space for the convolution kernel to move in fractional steps, which is why this is called a 'fractional stride' operation.

## Worked solution

We demonstrate zero insertion for stride-3 on a 1-D input of length 4.

**Input:** `[a, b, c, d]` (length 4).

**Output length formula:** `(4 - 1) * 3 + 1 = 10`.

**Output:** `[a, 0, 0, b, 0, 0, c, 0, 0, d]` — two zeros between every pair of original elements.

**Implementation:** Allocate a zeros tensor of shape `(10,)`. Then assign `output[::3] = input`, which scatters the original elements at positions 0, 3, 6, 9.

**Stride-1 case:** When `s = 1`, the formula gives `(L-1)*1 + 1 = L`, so the output equals the input. No zeros are inserted, which is correct — stride-1 ConvTranspose is a regular conv.

In [ ]:
import torch as t

t.manual_seed(6)

def zero_insert_1d(x: t.Tensor, s: int) -> t.Tensor:
    """Insert (s-1) zeros between each pair of adjacent elements in a 1-D tensor."""
    L = x.shape[0]
    L_out = (L - 1) * s + 1
    y = t.zeros(L_out, dtype=x.dtype)
    y[::s] = x
    return y

# s=3, input length 4
x = t.tensor([10.0, 20.0, 30.0, 40.0])
y = zero_insert_1d(x, s=3)
print(f"Input:  {x.tolist()}")
print(f"Output (s=3): {y.tolist()}")
print(f"Output length: {y.shape[0]} (expected {(4-1)*3+1} = 10)")
assert y.shape[0] == 10
assert y[0].item() == 10.0
assert y[3].item() == 20.0
assert y[6].item() == 30.0
assert y[9].item() == 40.0
assert (y[[1, 2, 4, 5, 7, 8]] == 0).all()

# s=1: no zeros inserted
y1 = zero_insert_1d(x, s=1)
print(f"Output (s=1): {y1.tolist()} (should equal input)")
assert t.equal(y1, x)

print("1-D zero insertion verified for s=1 and s=3.")